In [4]:
# Some basics:
import cv2
import math
import os
import pandas as pd


# Declare Filepaths
ROOT_DATA_DIR = (r"~\Data\Eyetracking_04_Match_Segmentation") # folder where the Input data sits
DATA_DIR_OUTPUT =  (r"~\Data\Eyetracking_05_Semantic_Categories")
segmentation_categories_path = (r"~\EyeTracking\Lib\category_mapping.xlsx")


# Extract the relevant index and semantic category name from the file.
segmentation_categories = pd.read_excel((segmentation_categories_path), index_col=0)
# print(segmentation_categories.head())
category_mapping = {
    idx: (name, category, designation)
    for idx, name, category, designation in zip(
        segmentation_categories['Idx'],
        segmentation_categories['Name'],
        segmentation_categories['Category'],
        segmentation_categories['Designation']
    )
}


# Define the column names for each type of eyetracking data
fixation_columns = ['tStart',	'tEnd',	'duration',	'xAvg',	'yAvg',	'pupilAvg',	'Movie', 
                    'matched_frame_IDx_start', 'matched_frame_IDx_end', 'OutOfBounds', 
                    'MatchingPixel_Start', 'MatchingPixel_End','MatchingPixel_Category']

saccade_columns  = ['tStart', 'tEnd', 'duration', 'xStart', 'yStart', 'xEnd', 'yEnd', 'ampDeg', 
                    'vPeak', 'Movie', 'matched_frame_IDx_start', 'matched_frame_IDx_end', 
                    'OutOfBounds', 'MatchingPixel_Start','MatchingPixel_End']

sample_columns   = ['tSample', 'LX', 'LY', 'LPupil', 'RX', 'RY', 'RPupil', 'Movie', 'matched_frame_IDx', 'MatchingPixel', 'OutOfBounds']


for pixel_categories_file in os.listdir(ROOT_DATA_DIR):  # Assuming your eye tracking files have a .xlsx extension
    # Construct the output file path
    output_file = os.path.join(DATA_DIR_OUTPUT, f'{pixel_categories_file[:-19]}_CategoryAssigned.csv')
    output_file_path = os.path.join(DATA_DIR_OUTPUT, output_file)

    # Check if the output file already exists
    if os.path.exists(output_file_path):
        #print(f"Output file already exists for {pixel_categories_file}. Skipping to the next file.")
        continue
    
    file_path = os.path.join(ROOT_DATA_DIR, pixel_categories_file)
    pixel_categories_file_data = pd.read_csv((file_path), delimiter=',')
    print('File Loaded', file_path)
    #print(pixel_categories_file_data.head())
    

    # Extract the information from the eyetracking file name
    file_parts = pixel_categories_file[:-4].split("_")
    participant = file_parts[0] 
    block_number = file_parts[-7]
    order_number = file_parts[-5]
    eyetracking_type = file_parts[-9]

    print('eyetracking_type',eyetracking_type)

    # Determine the column structure based on the eyetracking type
    if eyetracking_type == "Fixation":
        eyetracking_columns = fixation_columns
    elif eyetracking_type == "Saccade":
        eyetracking_columns = saccade_columns
    elif eyetracking_type == "Samples":
        eyetracking_columns = sample_columns
    else:
        print(f"Unknown eyetracking type in file: {pixel_categories_file_data}")
        continue

    # Iterate through the rows of the data file and add the matching category name
    if eyetracking_type == "Samples":
        if 'MatchingPixel' not in pixel_categories_file_data.columns:
            print("Column 'MatchingPixel' not found in the data file. Skipping to the next file.")
            continue

        pixel_categories_file_data['AssignedCategory'] = ''
        

        for index, row in pixel_categories_file_data.iterrows():
            pixel = row['MatchingPixel']
                        
            # Assign the category based on the index
            category = category_mapping.get(pixel,('', '', ''))
                        
            # Assign the category names to the respective columns
            pixel_categories_file_data.at[index, 'AssignedCategory'] = category[0]
            pixel_categories_file_data.at[index, 'Category'] = category[1]
            pixel_categories_file_data.at[index, 'Designation'] = category[2]
            
        # Save the modified data file
        pixel_categories_file_data.to_csv(output_file_path, index=True)
        print('File Saved')
        

    elif eyetracking_type == "Saccade":
        # Check if 'MatchingPixel_Start' and 'MatchingPixel_End' columns are present in the data
        if 'MatchingPixel_Start' not in pixel_categories_file_data.columns or 'MatchingPixel_End' not in pixel_categories_file_data.columns:
            print("Columns 'MatchingPixel_Start' or 'MatchingPixel_End' not found in the data file. Skipping to the next file.")
            continue

        pixel_categories_file_data['AssignedCategory_Start'] = ''
        pixel_categories_file_data['AssignedCategory_End'] = ''

        for index, row in pixel_categories_file_data.iterrows():
            start_pixel = row['MatchingPixel_Start']
            end_pixel = row['MatchingPixel_End']
            
            # Assign the category based on the index
            category_start = category_mapping.get(start_pixel, ('', '', ''))
            category_end = category_mapping.get(end_pixel, ('', '', ''))
            
            # Assign the category names to the respective columns
            pixel_categories_file_data.at[index, 'AssignedCategory_Start'] = category_start[0]
            pixel_categories_file_data.at[index, 'AssignedCategory_End'] = category_end[0]
            pixel_categories_file_data.at[index, 'Category_Start'] = category_start[1]
            pixel_categories_file_data.at[index, 'Category_End'] = category_end[1]
            pixel_categories_file_data.at[index, 'Designation_Start'] = category_start[2]
            pixel_categories_file_data.at[index, 'Designation_End'] = category_end[2]


        # Save the modified data file
        pixel_categories_file_data.to_csv(output_file_path, index=True)
        print('File Saved')


    elif eyetracking_type == "Fixation":
        # Check if 'MatchingPixel_Start' and 'MatchingPixel_End' columns are present in the data
        if 'MatchingPixel_Start' not in pixel_categories_file_data.columns or 'MatchingPixel_End' not in pixel_categories_file_data.columns:
            print("Columns 'MatchingPixel_Start' or 'MatchingPixel_End' not found in the data file. Skipping to the next file.")
            continue

        pixel_categories_file_data['AssignedCategory_Start'] = ''
        pixel_categories_file_data['AssignedCategory_End'] = ''

        for index, row in pixel_categories_file_data.iterrows():
            start_pixel = row['MatchingPixel_Start']
            end_pixel = row['MatchingPixel_End']
            
            # Assign the category based on the index
            category_start = category_mapping.get(start_pixel, ('', '', ''))
            category_end = category_mapping.get(end_pixel, ('', '', ''))
            
            # Assign the category names to the respective columns
            pixel_categories_file_data.at[index, 'AssignedCategory_Start'] = category_start[0]
            pixel_categories_file_data.at[index, 'AssignedCategory_End'] = category_end[0]
            pixel_categories_file_data.at[index, 'Category_Start'] = category_start[1]
            pixel_categories_file_data.at[index, 'Category_End'] = category_end[1]
            pixel_categories_file_data.at[index, 'Designation_Start'] = category_start[2]
            pixel_categories_file_data.at[index, 'Designation_End'] = category_end[2]


        # Save the modified data file
        pixel_categories_file_data.to_csv(output_file_path, index=True)
        print('File Saved')




FileNotFoundError: [WinError 3] The system cannot find the path specified: 'E:\\University\\Master\\4.SS22\\IMaC Lab\\Shamem\\EyeTracking\\Data\\Eyetracking_04_Match_Segmentation'